In [1]:
import random
import numpy as np
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [2]:
from datasets import load_dataset

dataset = load_dataset("glue", "sst2")

small_train = dataset["train"].shuffle(seed=42).select(range(3000))
small_val = dataset["validation"].shuffle(seed=42).select(range(500))

print("Train size:", len(small_train))
print("Validation size:", len(small_val))

print(small_train[0])

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report


X_train = small_train["sentence"]
y_train = small_train["label"]

X_val = small_val["sentence"]
y_val = small_val["label"]


tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)


tfidf_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_model.fit(X_train_tfidf, y_train)


tfidf_preds = tfidf_model.predict(X_val_tfidf)


tfidf_acc = accuracy_score(y_val, tfidf_preds)
tfidf_f1 = f1_score(y_val, tfidf_preds, average="macro")

print("TF-IDF + Logistic Regression Results")
print("Accuracy:", tfidf_acc)
print("Macro-F1:", tfidf_f1)
print()
print(classification_report(y_val, tfidf_preds, target_names=["negative", "positive"]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Train size: 3000
Validation size: 500
{'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1, 'idx': 32326}
TF-IDF + Logistic Regression Results
Accuracy: 0.76
Macro-F1: 0.7564935064935066

              precision    recall  f1-score   support

    negative       0.80      0.67      0.73       239
    positive       0.74      0.84      0.79       261

    accuracy                           0.76       500
   macro avg       0.77      0.76      0.76       500
weighted avg       0.76      0.76      0.76       500



In [3]:
# Second tokenization / preprocessing strategy

tfidf_vectorizer_2 = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 1)
)

X_train_tfidf_2 = tfidf_vectorizer_2.fit_transform(X_train)
X_val_tfidf_2 = tfidf_vectorizer_2.transform(X_val)

tfidf_model_2 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_model_2.fit(X_train_tfidf_2, y_train)

tfidf_preds_2 = tfidf_model_2.predict(X_val_tfidf_2)

tfidf_acc_2 = accuracy_score(y_val, tfidf_preds_2)
tfidf_f1_2 = f1_score(y_val, tfidf_preds_2, average="macro")

print("TF-IDF Strategy 2 + Logistic Regression Results")
print("Accuracy:", tfidf_acc_2)
print("Macro-F1:", tfidf_f1_2)
print()
print(classification_report(y_val, tfidf_preds_2, target_names=["negative", "positive"]))

TF-IDF Strategy 2 + Logistic Regression Results
Accuracy: 0.73
Macro-F1: 0.7170416388948275

              precision    recall  f1-score   support

    negative       0.84      0.54      0.66       239
    positive       0.68      0.90      0.78       261

    accuracy                           0.73       500
   macro avg       0.76      0.72      0.72       500
weighted avg       0.76      0.73      0.72       500



In [4]:
# Optional Choice for trigrams.
tfidf_vectorizer_3 = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    max_features=15000,
    ngram_range=(1, 3)
)

X_train_tfidf_3 = tfidf_vectorizer_3.fit_transform(X_train)
X_val_tfidf_3 = tfidf_vectorizer_3.transform(X_val)

tfidf_model_3 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_model_3.fit(X_train_tfidf_3, y_train)

tfidf_preds_3 = tfidf_model_3.predict(X_val_tfidf_3)

tfidf_acc_3 = accuracy_score(y_val, tfidf_preds_3)
tfidf_f1_3 = f1_score(y_val, tfidf_preds_3, average="macro")

print("TF-IDF Strategy 3 + Logistic Regression Results")
print("Accuracy:", tfidf_acc_3)
print("Macro-F1:", tfidf_f1_3)
print()
print(classification_report(y_val, tfidf_preds_3, target_names=["negative", "positive"]))

TF-IDF Strategy 3 + Logistic Regression Results
Accuracy: 0.75
Macro-F1: 0.7471590794163219

              precision    recall  f1-score   support

    negative       0.77      0.67      0.72       239
    positive       0.73      0.82      0.77       261

    accuracy                           0.75       500
   macro avg       0.75      0.75      0.75       500
weighted avg       0.75      0.75      0.75       500



In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# It has been observed that the terms 'no' and 'not' are critical for distinguishing between positive and negative sentiment.
#Retaining these words during the preprocessing phase is essential, as they serve as primary discriminators for the model's classification accuracy.

custom_stop_words_remove_negation = list(ENGLISH_STOP_WORDS | {"no", "not"})

tfidf_vectorizer_4 = TfidfVectorizer(
    lowercase=True,
    stop_words=custom_stop_words_remove_negation,
    max_features=10000,
    ngram_range=(1, 2)
)

X_train_tfidf_4 = tfidf_vectorizer_4.fit_transform(X_train)
X_val_tfidf_4 = tfidf_vectorizer_4.transform(X_val)

tfidf_model_4 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_model_4.fit(X_train_tfidf_4, y_train)

tfidf_preds_4 = tfidf_model_4.predict(X_val_tfidf_4)

tfidf_acc_4 = accuracy_score(y_val, tfidf_preds_4)
tfidf_f1_4 = f1_score(y_val, tfidf_preds_4, average="macro")

print("TF-IDF Strategy 4: remove no/not + Logistic Regression Results")
print("Accuracy:", tfidf_acc_4)
print("Macro-F1:", tfidf_f1_4)
print()
print(classification_report(y_val, tfidf_preds_4, target_names=["negative", "positive"]))

TF-IDF Strategy 4: remove no/not + Logistic Regression Results
Accuracy: 0.724
Macro-F1: 0.708299512987013

              precision    recall  f1-score   support

    negative       0.85      0.51      0.64       239
    positive       0.67      0.92      0.78       261

    accuracy                           0.72       500
   macro avg       0.76      0.72      0.71       500
weighted avg       0.76      0.72      0.71       500



In [6]:
import re
from collections import Counter

def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.split()

# Vocabulary oluşturma
counter = Counter()

for sentence in X_train:
    tokens = simple_tokenizer(sentence)
    counter.update(tokens)

# En sık geçen 10000 kelime
vocab_size = 10000
special_tokens = ["<PAD>", "<UNK>"]

vocab = {token: idx for idx, token in enumerate(special_tokens)}

for word, freq in counter.most_common(vocab_size - len(special_tokens)):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Example tokens:", simple_tokenizer(X_train[0]))

max_len = 64

def encode_sentence(sentence, vocab, max_len=64):
    tokens = simple_tokenizer(sentence)
    ids = []

    for token in tokens:
        ids.append(vocab.get(token, vocab["<UNK>"]))

    # Truncation
    ids = ids[:max_len]

    # Padding
    if len(ids) < max_len:
        ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))

    return ids

X_train_ids = [encode_sentence(sentence, vocab, max_len) for sentence in X_train]
X_val_ids = [encode_sentence(sentence, vocab, max_len) for sentence in X_val]

print("Original sentence:")
print(X_train[0])

print("\nEncoded sentence:")
print(X_train_ids[0])

print("\nLength:", len(X_train_ids[0]))

from torch.utils.data import Dataset, DataLoader
import torch

class SST2Dataset(Dataset):
    def __init__(self, encoded_texts, labels):
        self.encoded_texts = encoded_texts
        self.labels = labels

    def __len__(self):
        return len(self.encoded_texts)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.encoded_texts[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return input_ids, label


train_dataset = SST2Dataset(X_train_ids, y_train)
val_dataset = SST2Dataset(X_val_ids, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Number of train batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))

Vocabulary size: 5805
Example tokens: ['klein', 'charming', 'in', 'comedies', 'like', 'american', 'pie', 'and', 'deadon', 'in', 'election']
Original sentence:
klein , charming in comedies like american pie and dead-on in election , 

Encoded sentence:
[2567, 403, 10, 735, 34, 202, 2568, 4, 2569, 10, 2570, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Length: 64
Number of train batches: 94
Number of validation batches: 16


In [7]:
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pad_idx):
        super(BiLSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)

        outputs, (hidden, cell) = self.lstm(embedded)

        # Son forward ve backward hidden state'leri alıyoruz
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)

        logits = self.fc(final_hidden)

        return logits

In [8]:
embedding_dim = 100
hidden_dim = 128
output_dim = 2
pad_idx = vocab["<PAD>"]

bilstm_model = BiLSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    pad_idx=pad_idx
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=0.001)

print(bilstm_model)

def train_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for input_ids, labels in data_loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(input_ids)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(data_loader)


def evaluate_model(model, data_loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for input_ids, labels in data_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            outputs = model(input_ids)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return acc, macro_f1, all_preds, all_labels

num_epochs = 5

for epoch in range(num_epochs):
    train_loss = train_epoch(
        bilstm_model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_acc, val_f1, bilstm_preds, bilstm_labels = evaluate_model(
        bilstm_model,
        val_loader,
        device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Validation Macro-F1: {val_f1:.4f}")
    print("-" * 40)

BiLSTMClassifier(
  (embedding): Embedding(5805, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=2, bias=True)
)
Epoch 1/5
Train Loss: 0.6764
Validation Accuracy: 0.5940
Validation Macro-F1: 0.5913
----------------------------------------
Epoch 2/5
Train Loss: 0.6020
Validation Accuracy: 0.5940
Validation Macro-F1: 0.5774
----------------------------------------
Epoch 3/5
Train Loss: 0.4781
Validation Accuracy: 0.6140
Validation Macro-F1: 0.6025
----------------------------------------
Epoch 4/5
Train Loss: 0.3131
Validation Accuracy: 0.6520
Validation Macro-F1: 0.6511
----------------------------------------
Epoch 5/5
Train Loss: 0.1892
Validation Accuracy: 0.6060
Validation Macro-F1: 0.5878
----------------------------------------


In [10]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

model_name = "distilbert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return bert_tokenizer(
        example["sentence"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_bert = small_train.map(tokenize_function, batched=True)
val_bert = small_val.map(tokenize_function, batched=True)

train_bert = train_bert.remove_columns(["sentence", "idx"])
val_bert = val_bert.remove_columns(["sentence", "idx"])

train_bert = train_bert.rename_column("label", "labels")
val_bert = val_bert.rename_column("label", "labels")

train_bert.set_format("torch")
val_bert.set_format("torch")

print(train_bert[0])

import evaluate
from transformers import DataCollatorWithPadding

bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
).to(device)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )["accuracy"]

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["f1"]

    return {
        "accuracy": acc,
        "macro_f1": f1
    }

training_args = TrainingArguments(
    output_dir="./distilbert_sst2_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_bert,
    eval_dataset=val_bert,
    compute_metrics=compute_metrics
)

trainer.train()

bert_predictions = trainer.predict(val_bert)
bert_logits = bert_predictions.predictions
bert_preds = np.argmax(bert_logits, axis=1)

misclassified_examples = []

for i in range(len(small_val)):
    true_label = small_val[i]["label"]
    pred_label = int(bert_preds[i])

    if true_label != pred_label:
        misclassified_examples.append({
            "sentence": small_val[i]["sentence"],
            "true_label": "positive" if true_label == 1 else "negative",
            "predicted_label": "positive" if pred_label == 1 else "negative"
        })

misclassified_examples[:5]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'labels': tensor(1), 'input_ids': tensor([  101, 12555,  1010, 11951,  1999, 22092,  2066,  2137, 11345,  1998,
         2757,  1011,  2006,  1999,  2602,  1010,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.319180,0.338933,0.854000,0.853929
2,0.195752,0.364919,0.872000,0.871538


[{'sentence': "you 'll gasp appalled and laugh outraged and possibly , watching the spectacle of a promising young lad treading desperately in a nasty sea , shed an errant tear . ",
  'true_label': 'positive',
  'predicted_label': 'negative'},
 {'sentence': 'sam mendes has become valedictorian at the school for soft landings and easy ways out . ',
  'true_label': 'negative',
  'predicted_label': 'positive'},
 {'sentence': 'not far beneath the surface , this reconfigured tale asks disturbing questions about those things we expect from military epics . ',
  'true_label': 'positive',
  'predicted_label': 'negative'},
 {'sentence': "no screen fantasy-adventure in recent memory has the showmanship of clones ' last 45 minutes . ",
  'true_label': 'positive',
  'predicted_label': 'negative'},
 {'sentence': "i 'll bet the video game is a lot more fun than the film . ",
  'true_label': 'negative',
  'predicted_label': 'positive'}]